# MT5 AI Trader ? Kaggle One-Click Auto Train

Ch?y **cell b?n d??i** l? ??: clone/pull repo, install deps, t?m CSV trong `/kaggle/input`, auto-improve, verify report, copy model/report, zip output.

Kh?ng ch?y MT5/paper/demo/live tr?n Kaggle.


In [ ]:
# Kaggle one-click auto train: run this cell only
import os, sys, glob, json, shutil, subprocess, platform

REPO_URL = "https://github.com/quoccuongphan300992-cmd/mt5-ai-trader.git"
REPO_DIR = "/kaggle/working/mt5-ai-trader"
SYMBOL = "EURUSD"
TIMEFRAME = "H1"
BARS = 50000
LOCAL_CSV = "data/EURUSD_H1.csv"

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Kaggle inputs:", glob.glob("/kaggle/input/*"))

# 1) Clone/update repo
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "--oneline", "-1"], check=True)
print("CWD:", os.getcwd())

# 2) Install dependencies
req = "requirements-colab.txt" if os.path.exists("requirements-colab.txt") else "requirements.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req], check=True)

# 3) Find/copy CSV from Kaggle Input
os.makedirs("data", exist_ok=True)
if not os.path.exists(LOCAL_CSV):
    candidates = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    print("CSV candidates:")
    for path in candidates:
        print(" -", path)
    if not candidates:
        raise FileNotFoundError("No CSV found under /kaggle/input. Add your CSV dataset via Add Input.")
    preferred = [x for x in candidates if "EURUSD" in os.path.basename(x).upper() and "H1" in os.path.basename(x).upper()]
    source_csv = preferred[0] if preferred else candidates[0]
    shutil.copy2(source_csv, LOCAL_CSV)
    print("Using CSV:", source_csv)
print("LOCAL_CSV:", os.path.abspath(LOCAL_CSV), os.path.getsize(LOCAL_CSV), "bytes")

# 4) Run auto-improve offline search
cmd = [
    sys.executable, "main.py", "auto-improve",
    "--csv", LOCAL_CSV,
    "--symbol", SYMBOL,
    "--timeframe", TIMEFRAME,
    "--bars", str(BARS),
    "--max-rounds", "5",
    "--folds", "2",
    "--min-trades", "60",
    "--min-profit-factor", "1.05",
    "--min-expectancy", "0.0",
    "--min-positive-fold-ratio", "1.0",
    "--max-drawdown-limit", "0.20",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

# 5) Verify best report
best_path = "reports/auto_improve_best.json"
if not os.path.exists(best_path):
    raise FileNotFoundError("auto_improve_best.json missing; auto-improve did not complete")
best = json.load(open(best_path, encoding="utf-8"))
print(json.dumps(best, indent=2))
print("candidate_pass:", best.get("candidate_pass"))
print("production_artifacts_written:", best.get("production_artifacts_written"))

# 6) Copy and zip outputs for Kaggle download
out_dir = "/kaggle/working/output"
for sub in ["models", "reports"]:
    os.makedirs(f"{out_dir}/{sub}", exist_ok=True)
    if os.path.exists(sub):
        for item in glob.glob(f"{sub}/*"):
            dst = f"{out_dir}/{sub}/{os.path.basename(item)}"
            if os.path.isdir(item):
                shutil.copytree(item, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(item, dst)

zip_path = "/kaggle/working/mt5_ai_outputs_kaggle.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive(zip_path[:-4], "zip", out_dir)

print("=== OUTPUT FILES ===")
for path in sorted(glob.glob(f"{out_dir}/**/*", recursive=True)):
    if os.path.isfile(path):
        print(path)
print("ZIP:", zip_path, os.path.getsize(zip_path), "bytes")
